# BirdCLEF+ 2026 — Exploratory Data Analysis

This notebook provides a systematic exploration of the BirdCLEF+ 2026 dataset.  
The goal is to understand the data distribution, audio characteristics, and class imbalance  
before building the classification pipeline.

**Dataset**: 234 wildlife species (birds, amphibians, insects, mammals, reptiles)  
**Audio**: 32 kHz OGG recordings from Xeno-canto and iNaturalist  
**Metric**: Macro-averaged ROC-AUC (skipping classes with no positives)

---

**Sections**
1. Environment setup
2. Dataset overview
3. Species distribution
4. Recording quality and metadata
5. Audio duration analysis
6. Geographic distribution
7. Waveform and mel spectrogram visualization
8. Class imbalance analysis
9. Key findings and implications

## 1. Environment Setup

In [ ]:
import os
import warnings
from pathlib import Path

import librosa
import librosa.display
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import pandas as pd
import seaborn as sns
import soundfile as sf
from IPython.display import Audio, display

warnings.filterwarnings("ignore")

# Consistent plot style throughout
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})
sns.set_palette("muted")

# Paths — adjust DATA_ROOT to your local data location
DATA_ROOT = Path("../data/raw")
TRAIN_CSV = DATA_ROOT / "train.csv"
TAXONOMY_CSV = DATA_ROOT / "taxonomy.csv"
TRAIN_AUDIO = DATA_ROOT / "train_audio"
TRAIN_SOUNDSCAPES = DATA_ROOT / "train_soundscapes"

SAMPLE_RATE = 32000  # competition standard

print(f"Data root: {DATA_ROOT.resolve()}")
print(f"train.csv exists: {TRAIN_CSV.exists()}")
print(f"taxonomy.csv exists: {TAXONOMY_CSV.exists()}")

## 2. Dataset Overview

In [ ]:
# Load the main training metadata
df = pd.read_csv(TRAIN_CSV)
taxonomy = pd.read_csv(TAXONOMY_CSV)

print("=" * 50)
print("TRAINING SET")
print("=" * 50)
print(f"Total recordings  : {len(df):,}")
print(f"Unique species    : {df['primary_label'].nunique()}")
print(f"Unique authors    : {df['author'].nunique():,}")
print(f"Collections       : {df['collection'].value_counts().to_dict()}")
print()
print("=" * 50)
print("TAXONOMY")
print("=" * 50)
print(taxonomy["class_name"].value_counts().to_string())
print(f"\nTotal species in submission: {len(taxonomy)}")

In [ ]:
# Column overview
print("Columns and dtypes:")
print(df.dtypes)
print()
print("Missing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
# Sample rows to understand structure
df[["primary_label", "secondary_labels", "filename",
    "rating", "collection", "latitude", "longitude"]].head(10)

## 3. Species Distribution

In [ ]:
# Recordings per species — sorted descending
species_counts = df["primary_label"].value_counts()

print(f"Mean recordings per species  : {species_counts.mean():.1f}")
print(f"Median recordings per species: {species_counts.median():.0f}")
print(f"Min recordings               : {species_counts.min()} ({species_counts.idxmin()})")
print(f"Max recordings               : {species_counts.max()} ({species_counts.idxmax()})")
print(f"Species with < 10 recordings : {(species_counts < 10).sum()}")
print(f"Species with < 5 recordings  : {(species_counts < 5).sum()}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: Distribution of recordings per species
axes[0].hist(species_counts.values, bins=50, color="steelblue", edgecolor="white", linewidth=0.5)
axes[0].set_xlabel("Recordings per species")
axes[0].set_ylabel("Number of species")
axes[0].set_title("Distribution of recordings per species")
axes[0].axvline(species_counts.median(), color="firebrick", linestyle="--",
                linewidth=1.5, label=f"Median = {species_counts.median():.0f}")
axes[0].legend()

# Right: Top 30 and bottom 30 species by recording count
top30 = species_counts.head(30)
axes[1].barh(range(30), top30.values[::-1], color="steelblue")
axes[1].set_yticks(range(30))
axes[1].set_yticklabels(top30.index[::-1], fontsize=8)
axes[1].set_xlabel("Number of recordings")
axes[1].set_title("Top 30 species by recording count")

plt.tight_layout()
plt.savefig("../experiments/species_distribution.png", bbox_inches="tight")
plt.show()

In [ ]:
# Recordings per taxonomic class
df_tax = df.merge(taxonomy[["primary_label", "class_name"]], on="primary_label", how="left")

class_counts = df_tax["class_name"].value_counts()

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(class_counts.index, class_counts.values, color="steelblue", edgecolor="white")

# Annotate bars with counts
for bar, count in zip(bars, class_counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 20,
            f"{count:,}", ha="center", va="bottom", fontsize=10)

ax.set_xlabel("Taxonomic class")
ax.set_ylabel("Number of recordings")
ax.set_title("Recording count by taxonomic class")
plt.tight_layout()
plt.show()

print(class_counts.to_string())

## 4. Recording Quality and Metadata

In [ ]:
# Xeno-canto quality ratings (1-5; 0 = no rating / iNat recordings)
rating_counts = df["rating"].value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Rating distribution
axes[0].bar(rating_counts.index.astype(str), rating_counts.values,
            color="steelblue", edgecolor="white")
axes[0].set_xlabel("Rating (0 = not rated / iNat)")
axes[0].set_ylabel("Number of recordings")
axes[0].set_title("XC quality rating distribution")

# XC vs iNat split
coll_counts = df["collection"].value_counts()
axes[1].pie(coll_counts.values, labels=coll_counts.index,
            autopct="%1.1f%%", colors=["steelblue", "coral"],
            startangle=90, wedgeprops={"edgecolor": "white", "linewidth": 1.5})
axes[1].set_title("Recording collection source")

plt.tight_layout()
plt.show()

print(f"Recordings with rating >= 4 (high quality): {(df['rating'] >= 4).sum():,}")
print(f"Recordings with secondary labels          : {df['secondary_labels'].notna().sum():,}")

In [ ]:
# Secondary label analysis
# Secondary labels indicate background species — important for multi-label training
has_secondary = df["secondary_labels"].dropna()
has_secondary = has_secondary[has_secondary != "[]"]

n_secondary = has_secondary.apply(
    lambda x: len(str(x).split()) if isinstance(x, str) else 0
)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(n_secondary.values, bins=20, color="steelblue", edgecolor="white")
ax.set_xlabel("Number of secondary species per recording")
ax.set_ylabel("Count")
ax.set_title("Distribution of secondary (background) species counts")
plt.tight_layout()
plt.show()

print(f"Recordings with secondary labels: {len(has_secondary):,}")
print(f"Mean secondary species per recording: {n_secondary.mean():.2f}")

## 5. Audio Duration Analysis

In [ ]:
# Sample 500 files to estimate duration distribution without loading everything
# Full computation would take too long during EDA

sample_files = df["filename"].sample(n=min(500, len(df)), random_state=42)
durations = []

for fname in sample_files:
    fpath = TRAIN_AUDIO / fname
    if not fpath.exists():
        continue
    try:
        info = sf.info(str(fpath))
        duration_s = info.frames / info.samplerate
        durations.append(duration_s)
    except Exception:
        pass

durations = np.array(durations)
print(f"Files sampled      : {len(durations)}")
print(f"Mean duration      : {durations.mean():.1f}s")
print(f"Median duration    : {np.median(durations):.1f}s")
print(f"Min duration       : {durations.min():.1f}s")
print(f"Max duration       : {durations.max():.1f}s")
print(f"Duration < 5s      : {(durations < 5).sum()} ({100*(durations<5).mean():.1f}%)")
print(f"Duration >= 5s     : {(durations >= 5).sum()} ({100*(durations>=5).mean():.1f}%)")

In [ ]:
if len(durations) > 0:
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.hist(np.clip(durations, 0, 60), bins=60, color="steelblue", edgecolor="white")
    ax.axvline(5, color="firebrick", linestyle="--", linewidth=1.5,
               label="5s window (competition unit)")
    ax.set_xlabel("Duration (seconds, clipped at 60s)")
    ax.set_ylabel("Number of recordings")
    ax.set_title("Audio duration distribution (sample of 500 recordings)")
    ax.legend()
    plt.tight_layout()
    plt.show()

## 6. Geographic Distribution

In [ ]:
# XC recordings have coordinates; iNat recordings may not
geo_df = df.dropna(subset=["latitude", "longitude"]).copy()

print(f"Recordings with coordinates: {len(geo_df):,} / {len(df):,}")
print(f"Latitude  range: {geo_df['latitude'].min():.2f} to {geo_df['latitude'].max():.2f}")
print(f"Longitude range: {geo_df['longitude'].min():.2f} to {geo_df['longitude'].max():.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

# Color by collection source
for coll, color in [("XC", "steelblue"), ("iNat", "coral")]:
    subset = geo_df[geo_df["collection"] == coll]
    ax.scatter(subset["longitude"], subset["latitude"],
               alpha=0.3, s=5, color=color, label=coll)

# Highlight Pantanal region (approx bounding box)
from matplotlib.patches import Rectangle
pantanal = Rectangle((-60, -22), 12, 12, linewidth=2,
                     edgecolor="darkgreen", facecolor="none",
                     linestyle="--", label="Pantanal region (approx)")
ax.add_patch(pantanal)

ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Geographic distribution of training recordings")
ax.legend(markerscale=4)
plt.tight_layout()
plt.show()

# Pantanal subset
pantanal_df = geo_df[
    (geo_df["latitude"].between(-22, -10)) &
    (geo_df["longitude"].between(-60, -48))
]
print(f"Recordings within Pantanal bounding box: {len(pantanal_df):,}")

## 7. Waveform and Mel Spectrogram Visualization

In [ ]:
def load_and_show(filepath, label="", sr=32000):
    """
    Load one audio file, display waveform and mel spectrogram side by side.
    Also renders an interactive audio player inline.
    """
    waveform, _ = librosa.load(str(filepath), sr=sr, mono=True)

    # Crop to first 5 seconds for display
    clip = waveform[:5 * sr]

    # Mel spectrogram parameters matching the training pipeline
    mel = librosa.feature.melspectrogram(
        y=clip, sr=sr, n_fft=1024, hop_length=320,
        n_mels=128, fmin=50, fmax=14000, power=2.0
    )
    mel_db = librosa.power_to_db(mel, ref=np.max, top_db=80)

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    fig.suptitle(label, fontsize=12, fontweight="bold")

    # Waveform
    times = np.linspace(0, len(clip) / sr, len(clip))
    axes[0].plot(times, clip, color="steelblue", linewidth=0.6)
    axes[0].set_xlabel("Time (s)")
    axes[0].set_ylabel("Amplitude")
    axes[0].set_title("Waveform (first 5 seconds)")

    # Mel spectrogram
    img = librosa.display.specshow(
        mel_db, sr=sr, hop_length=320, x_axis="time", y_axis="mel",
        fmin=50, fmax=14000, ax=axes[1], cmap="magma"
    )
    fig.colorbar(img, ax=axes[1], format="%+2.0f dB")
    axes[1].set_title("Log-mel spectrogram (128 mel bins, 50-14000 Hz)")

    plt.tight_layout()
    plt.show()

    # Interactive playback
    display(Audio(clip, rate=sr))

    return waveform

print("Function defined. Running on example files below.")

In [ ]:
# Visualize one recording from each taxonomic class
# This shows how different species look in the spectrogram domain

classes_to_show = ["Aves", "Amphibia", "Insecta", "Mammalia", "Reptilia"]

for cls in classes_to_show:
    subset = df_tax[df_tax["class_name"] == cls]
    if len(subset) == 0:
        continue

    # Pick the highest-rated recording available
    row = subset.sort_values("rating", ascending=False).iloc[0]
    fpath = TRAIN_AUDIO / row["filename"]

    if not fpath.exists():
        print(f"{cls}: file not found — {fpath}")
        continue

    label = f"{cls} | {row['primary_label']} | Rating: {row['rating']}"
    load_and_show(fpath, label=label)

In [ ]:
# Compare mel spectrogram vs PCEN for the same recording
# PCEN is an alternative frontend with better noise robustness for PAM data

sample_row = df_tax[df_tax["class_name"] == "Aves"].sort_values("rating", ascending=False).iloc[0]
sample_path = TRAIN_AUDIO / sample_row["filename"]

if sample_path.exists():
    waveform, _ = librosa.load(str(sample_path), sr=SAMPLE_RATE, mono=True)
    clip = waveform[:5 * SAMPLE_RATE]

    mel = librosa.feature.melspectrogram(
        y=clip, sr=SAMPLE_RATE, n_fft=1024, hop_length=320,
        n_mels=128, fmin=50, fmax=14000, power=2.0
    )
    mel_db = librosa.power_to_db(mel, ref=np.max, top_db=80)

    mel_mag = librosa.feature.melspectrogram(
        y=clip, sr=SAMPLE_RATE, n_fft=1024, hop_length=320,
        n_mels=128, fmin=50, fmax=14000, power=1.0  # magnitude for PCEN
    )
    pcen_spec = librosa.pcen(mel_mag * (2 ** 31), sr=SAMPLE_RATE, hop_length=320)

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    fig.suptitle("Log-mel spectrogram vs PCEN — same recording", fontsize=12)

    librosa.display.specshow(
        mel_db, sr=SAMPLE_RATE, hop_length=320, x_axis="time", y_axis="mel",
        fmin=50, fmax=14000, ax=axes[0], cmap="magma"
    )
    axes[0].set_title("Log-mel spectrogram (dB)")

    librosa.display.specshow(
        pcen_spec, sr=SAMPLE_RATE, hop_length=320, x_axis="time", y_axis="mel",
        fmin=50, fmax=14000, ax=axes[1], cmap="magma"
    )
    axes[1].set_title("PCEN (per-channel energy normalization)")

    plt.tight_layout()
    plt.show()

    print("PCEN compresses dynamic range and suppresses stationary noise,")
    print("which may help with variable background noise in PAM recordings.")

## 8. Class Imbalance Analysis

In [ ]:
# Gini coefficient — measures inequality in class sizes
# 0 = perfectly balanced, 1 = maximally imbalanced

def gini(counts):
    counts = np.sort(np.array(counts, dtype=float))
    n = len(counts)
    index = np.arange(1, n + 1)
    return (2 * np.sum(index * counts) / (n * counts.sum())) - (n + 1) / n

g = gini(species_counts.values)
print(f"Gini coefficient (class imbalance): {g:.4f}")
print(f"  -> 0 = perfectly balanced, 1 = maximally imbalanced")

# Imbalance ratio: max / min
ratio = species_counts.max() / species_counts.min()
print(f"Imbalance ratio (max/min)          : {ratio:.1f}x")

In [ ]:
# Lorenz curve — visual representation of class imbalance
sorted_counts = np.sort(species_counts.values)
cumulative = np.cumsum(sorted_counts) / sorted_counts.sum()
n_species = len(sorted_counts)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Lorenz curve
axes[0].plot(np.linspace(0, 1, n_species), cumulative,
             color="steelblue", linewidth=2, label="Lorenz curve")
axes[0].plot([0, 1], [0, 1], color="gray", linestyle="--",
             linewidth=1, label="Perfect equality")
axes[0].fill_between(np.linspace(0, 1, n_species), cumulative,
                     np.linspace(0, 1, n_species), alpha=0.15, color="steelblue")
axes[0].set_xlabel("Cumulative fraction of species (sorted by count)")
axes[0].set_ylabel("Cumulative fraction of recordings")
axes[0].set_title(f"Lorenz curve (Gini = {g:.3f})")
axes[0].legend()

# Bucket analysis: how many species fall in each bin
bins = [0, 5, 10, 20, 50, 100, 200, 500, 10000]
labels_bins = ["<5", "5-10", "10-20", "20-50", "50-100", "100-200", "200-500", "500+"]
bucket_counts = pd.cut(species_counts, bins=bins, labels=labels_bins).value_counts().sort_index()
axes[1].bar(bucket_counts.index, bucket_counts.values, color="steelblue", edgecolor="white")
axes[1].set_xlabel("Recordings per species (bucket)")
axes[1].set_ylabel("Number of species")
axes[1].set_title("Species count distribution by recording bucket")
for i, (idx, v) in enumerate(bucket_counts.items()):
    axes[1].text(i, v + 0.5, str(v), ha="center", fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# Implication for macro-AUC metric:
# Classes with very few samples will have high variance in AUC estimation.
# The competition metric skips classes with no positives in the eval set,
# but rare classes still need to be handled during training.

rare_species = species_counts[species_counts < 10]
common_species = species_counts[species_counts >= 50]

print(f"Rare species   (< 10 recordings) : {len(rare_species)} species")
print(f"Common species (>= 50 recordings): {len(common_species)} species")
print()
print("Strategies for rare species:")
print("  1. Pseudo-labelling from train_soundscapes")
print("  2. Focal loss to up-weight hard examples")
print("  3. Mixup augmentation between rare and common classes")
print("  4. Transfer from Perch embeddings (pre-trained on global bird data)")

## 9. Key Findings and Implications

This section summarises the most important observations from the EDA  
and their direct implications for model design.

In [ ]:
findings = {
    "Total training recordings": f"{len(df):,}",
    "Species in competition": 234,
    "Class imbalance (Gini)": f"{g:.3f}",
    "Imbalance ratio (max/min)": f"{ratio:.0f}x",
    "Species with < 10 recordings": len(rare_species),
    "Median duration (sample)": f"{np.median(durations):.1f}s" if len(durations) > 0 else "N/A",
    "Dominant collection": df['collection'].value_counts().idxmax(),
    "Dominant taxonomic class": df_tax['class_name'].value_counts().idxmax(),
}

print("Summary of Key Findings")
print("=" * 45)
for k, v in findings.items():
    print(f"  {k:<38}: {v}")

print()
print("Design Implications")
print("=" * 45)
implications = [
    "High class imbalance -> use focal loss or label smoothing",
    "Many short recordings -> pad by tiling, not zero-padding",
    "Pantanal domain gap -> pseudo-labels from train_soundscapes",
    "Rare classes -> Perch embeddings transfer well across species",
    "Macro-AUC metric -> per-class calibration matters more than accuracy",
    "CPU-only inference -> ONNX + batch windowing essential",
]
for imp in implications:
    print(f"  - {imp}")